# EXP-001 — Notebook làm việc

Hồ sơ chốt: `experiments/01_baselines/EXP-001_Dummy/runs/20261114/experiment.ipynb`. Mặc định chỉ đọc kết quả. Bật RUN_TRAIN khi cần chạy lại; output được lưu riêng. Nhật ký duy nhất là `experiments/experiments_logs.xlsx`.

In [1]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/wesad/models.py").is_file())
os.chdir(ROOT)
RUN_TRAIN = False
FOLDER = ROOT / 'experiments/01_baselines/EXP-001_Dummy/runs/20261114'
def read_csv(path, *args, **kwargs):
    path = Path(path)
    if not path.exists() and path.name == "metrics_by_subject.csv":
        path = path.with_name("results.csv")
    return pd.read_csv(path, *args, **kwargs)

In [2]:
if RUN_TRAIN:
    from pathlib import Path
    import os
    import sys

    ROOT = next(
        (
            folder
            for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder / "src" / "wesad" / "models.py").is_file()
        ),
        None,
    )

    if ROOT is None:
        raise RuntimeError("Không tìm thấy thư mục dự án.")

    os.chdir(ROOT)

    src_path = str(ROOT / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

    print("Project:", ROOT)
    print("Python:", sys.executable)

In [3]:
if RUN_TRAIN:
    import pandas as pd
    from IPython.display import display

    from wesad.data import SUBJECTS, load_feature_table
    from wesad.utils import read_config, file_hash


In [4]:
if RUN_TRAIN:
    defaults = read_config(ROOT / "configs" / "defaults.yaml")

    FEATURES = ROOT / defaults["features"]
    SCHEMA = ROOT / defaults["schema"]

    print("File dữ liệu:", FEATURES)
    print("File schema:", SCHEMA)


In [5]:
if RUN_TRAIN:
    assert FEATURES.is_file(), f"Không tìm thấy dữ liệu: {FEATURES}"
    assert SCHEMA.is_file(), f"Không tìm thấy schema: {SCHEMA}"

In [6]:
if RUN_TRAIN:
    actual_sha = file_hash(FEATURES)
    assert actual_sha == defaults["feature_sha256"], (
        f"Checksum không khớp. Giá trị hiện tại: {actual_sha}"
    )


In [7]:
if RUN_TRAIN:
    df, feature_columns = load_feature_table(
        FEATURES,
        SCHEMA,
        expected_subjects=SUBJECTS,
    )

    assert len(feature_columns) == 68, "Số đặc trưng khác 68."

    print("\nĐã kiểm tra checksum và cấu trúc dữ liệu.")
    print("Số cửa sổ:", len(df))
    print("Số đặc trưng:", len(feature_columns))
    print("Số người tham gia:", df["subject_id"].nunique())
    print("Số giá trị thiếu trong các đặc trưng:",
          int(df[feature_columns].isna().sum().sum()))

    print("\n5 dòng đầu:")
    display(df.head())

    print("\nSố cửa sổ theo người và nhãn:")
    display(
        pd.crosstab(
            df["subject_id"],
            df["label_original"],
        ).reindex(SUBJECTS)
    )

## R02 — Dummy baseline đủ 15 người

Mục tiêu: xác định mức đối chứng trên toàn bộ 15 người.

Giữ nguyên mô hình, dữ liệu và thiết lập của R01.
Mỗi fold train trên 14 người và test người còn lại.
Thực hiện hai bài toán multiclass và binary: tổng cộng 30 fold.

Giả thuyết: Dummy chỉ dự đoán lớp phổ biến nhất trong train,
nên không phân biệt được đầy đủ các trạng thái.

In [8]:
if RUN_TRAIN:
    from copy import deepcopy
    from wesad.evaluation import run_experiment

    CONFIG_R02 = deepcopy(defaults)

    CONFIG_R02.update({
        "protocol": "notebook_research_new_runs_v1",
        "experiment_id": "EXP-001",
        "group_experiment_id": "EXP-001",
        "run_no": "r02",
        "run_scope": "main",
        "models": ["dummy"],
        "tasks": ["multiclass", "binary"],
        "balance": False,
        "nested": False,
        "output_dir": "outputs/experiments/EXP-001/clean_v1/r02_main",
        "source_paths": [
            "src/wesad/data.py",
            "src/wesad/evaluation.py",
            "src/wesad/models.py",
            "src/wesad/utils.py",
        ],
    })

    RUN_DIR_R02 = ROOT / CONFIG_R02["output_dir"]

    assert file_hash(FEATURES) == CONFIG_R02["feature_sha256"]

    result_r02 = run_experiment(
        CONFIG_R02,
        resume=True,
        outer_subjects=None,
    )

    print("Trạng thái:", result_r02["status"])
    print("Kết quả:", RUN_DIR_R02)

In [9]:
if RUN_TRAIN:
    metrics_r02 = read_csv(
        RUN_DIR_R02 / "metrics_by_subject.csv"
    )



# Kiểm tra đủ 15 người × 2 bài toán.


In [10]:
if RUN_TRAIN:
    assert result_r02["status"] == "complete"
    assert len(metrics_r02) == 30
    assert set(metrics_r02["model"]) == {"dummy"}
    assert not metrics_r02.duplicated(
        ["task", "model", "subject_id"]
    ).any()

    for task in ["multiclass", "binary"]:
        part = metrics_r02.loc[metrics_r02["task"].eq(task)]
        assert set(part["subject_id"]) == set(SUBJECTS)

    print("PASS: đủ 30 fold, không trùng kết quả.")

    display(metrics_r02)



# Trung bình và độ lệch chuẩn giữa 15 người.


In [11]:
if RUN_TRAIN:
    summary_r02 = metrics_r02.groupby("task")[
        ["macro_f1", "balanced_accuracy"]
    ].agg(["mean", "std"])

    display(summary_r02)

## Kết quả và nhận xét R02

Đã hoàn thành LOSO cho 15 người trên hai bài toán, tổng cộng 30 fold.

- Binary: Macro-F1 = 0.412281 ± 0.003526;
  Balanced Accuracy = 0.500000 ± 0.
- Multiclass: Macro-F1 = 0.235047 ± 0.002305;
  Balanced Accuracy = 0.333333 ± 0.

Các giá trị được tổng hợp bằng trung bình và độ lệch chuẩn
giữa 15 người.

Dummy chỉ dự đoán lớp phổ biến nhất trong train.
Balanced Accuracy bằng 1/2 ở binary và 1/3 ở multiclass
phù hợp với việc chỉ nhận diện được một lớp.

Macro-F1 thay đổi nhẹ giữa các người do tỷ lệ nhãn khác nhau.
Độ lệch chuẩn thấp ở đây không chứng minh mô hình nhận diện tốt.

Kết quả này được dùng làm đối chứng cho các mô hình tiếp theo.

## Quyết định tiếp theo

Thực hiện EXP-002 — Logistic Regression, giữ nguyên dữ liệu
và cách đánh giá để so sánh với Dummy.

## Biểu đồ kết quả


In [12]:
if RUN_TRAIN:
    from pathlib import Path
    import os
    os.environ.setdefault("MPLCONFIGDIR", str(Path(os.environ["TEMP"]) / "wesad_matplotlib"))
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display, Image
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "src/wesad/models.py").is_file()), None)
    if ROOT is None:
        raise RuntimeError("Hãy mở notebook trong thư mục dự án.")

    RUN_DIR_VIS = ROOT / "outputs/experiments/EXP-001/clean_v1/r02_main"
    REFERENCE_VIS = None
    MODEL_LABEL = "Dummy"
    REFERENCE_LABEL = ""
    FIGURES_VIS = RUN_DIR_VIS / "figures"

    for name in ["metrics_by_subject.csv", "predictions.csv"]:
        if not (RUN_DIR_VIS / name).is_file():
            raise FileNotFoundError(f"Thiếu kết quả: {RUN_DIR_VIS / name}")

    metrics_vis = read_csv(RUN_DIR_VIS / "metrics_by_subject.csv")
    predictions_vis = read_csv(RUN_DIR_VIS / "predictions.csv")
    assert not metrics_vis.duplicated(["task", "model", "subject_id"]).any()
    assert not predictions_vis.duplicated(["task", "model", "subject_id", "window_id"]).any()
    assert metrics_vis.model.nunique() == 1
    reference_vis = None
    if REFERENCE_VIS is not None:
        reference_vis = read_csv(REFERENCE_VIS / "metrics_by_subject.csv")
        paired_vis = metrics_vis.merge(reference_vis, on=["task", "subject_id"],
                                       suffixes=("_current", "_reference"), validate="one_to_one")
        assert len(paired_vis) == len(metrics_vis)
        assert paired_vis.n_test_current.eq(paired_vis.n_test_reference).all()

    FIGURES_VIS.mkdir(parents=True, exist_ok=True)
    plt.switch_backend("Agg")
    def show_figure(fig, filename):
        fig.tight_layout()
        target = FIGURES_VIS / filename
        fig.savefig(target, dpi=150, bbox_inches="tight")
        plt.close(fig)
        display(Image(filename=str(target)))

    summary_vis = metrics_vis.groupby("task")[["macro_f1", "balanced_accuracy"]].agg(["mean", "std"])
    display(summary_vis)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric in zip(axes, ["macro_f1", "balanced_accuracy"]):
        means = summary_vis[(metric, "mean")]
        deviations = summary_vis[(metric, "std")].fillna(0)
        ax.bar(means.index, means, yerr=deviations, capsize=5, color=["#2c7fb8", "#41ab5d"][:len(means)])
        ax.set(title=metric.replace("_", " ").title(), ylim=(0, 1.05), ylabel="Điểm")
        ax.grid(axis="y", alpha=0.2)
        for x, mean in enumerate(means):
            ax.text(x, mean + 0.02, f"{mean:.3f}", ha="center")
    fig.suptitle(f"EXP-001 — {MODEL_LABEL}")
    show_figure(fig, "summary_metrics.png")

    for task in ["binary", "multiclass"]:
        part = metrics_vis.loc[metrics_vis.task.eq(task)].copy()
        if part.empty:
            continue
        subjects = sorted(part.subject_id.unique(), key=lambda s: int(s[1:]))
        part = part.set_index("subject_id").reindex(subjects)
        fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
        for ax, metric in zip(axes, ["macro_f1", "balanced_accuracy"]):
            scores = pd.DataFrame({MODEL_LABEL: part[metric]})
            if reference_vis is not None:
                ref = reference_vis.loc[reference_vis.task.eq(task)].set_index("subject_id").reindex(subjects)
                scores[REFERENCE_LABEL] = ref[metric]
            scores.plot.bar(ax=ax, ylim=(0, 1.05), rot=0)
            ax.set_ylabel(metric.replace("_", " ").title())
            ax.grid(axis="y", alpha=0.2)
        axes[0].set_title(f"EXP-001 — {task}: kết quả theo người test")
        axes[1].set_xlabel("Người test")
        show_figure(fig, f"{task}_subject_metrics.png")

        pred = predictions_vis.loc[predictions_vis.task.eq(task)]
        assert len(pred) == int(part.n_test.sum())
        labels = [0, 1] if task == "binary" else [1, 2, 3]
        names = ["Non-stress", "Stress"] if task == "binary" else ["Baseline", "Stress", "Amusement"]
        cm = confusion_matrix(pred.y_true, pred.y_pred, labels=labels)
        row_sums = cm.sum(axis=1, keepdims=True)
        normalized = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=axes[0], cmap="Blues", colorbar=False, values_format="d")
        ConfusionMatrixDisplay(normalized, display_labels=names).plot(ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f")
        axes[0].set_title("Số cửa sổ")
        axes[1].set_title("Tỷ lệ theo nhãn thật")
        fig.suptitle(f"EXP-001 — {task}: confusion matrix")
        show_figure(fig, f"{task}_confusion_matrix.png")

    if reference_vis is not None:
        fig, axes = plt.subplots(2, 1, figsize=(11, 6))
        for ax, task in zip(axes, ["binary", "multiclass"]):
            part = paired_vis.loc[paired_vis.task.eq(task)].copy()
            part["subject_number"] = part.subject_id.str[1:].astype(int)
            part = part.sort_values("subject_number")
            delta = part.macro_f1_current - part.macro_f1_reference
            ax.bar(part.subject_id, delta, color=["#2c7fb8" if d >= 0 else "#e6550d" for d in delta])
            ax.axhline(0, color="black", linewidth=0.8)
            ax.set(title=task, ylabel="Chênh lệch Macro-F1")
            ax.grid(axis="y", alpha=0.2)
        fig.suptitle(f"{MODEL_LABEL} so với {REFERENCE_LABEL}")
        show_figure(fig, "comparison_delta_macro_f1.png")

    print("Ảnh được lưu tại:", FIGURES_VIS)
    if metrics_vis.subject_id.nunique() == 1:
        print("Lượt kiểm tra một người; chưa dùng để kết luận cho đủ 15 người.")
    else:
        print("Cột sai số ở bảng tổng hợp là độ lệch chuẩn giữa các người.")
    print("Confusion matrix gộp cửa sổ; bảng tổng hợp tính trung bình theo người.")


In [13]:
saved = pd.read_csv(FOLDER / "results.csv")
assert len(saved) == 30
display(saved.groupby("task")[["macro_f1", "balanced_accuracy"]].agg(["mean", "std"]))

            macro_f1           balanced_accuracy     
                mean       std              mean  std
task                                                 
binary      0.412281  0.003526          0.500000  0.0
multiclass  0.235047  0.002305          0.333333  0.0
